# Week 1, Lab 2 — Structured output (JSON you can trust)

**Course:** Agentic AI Engineering — Local Models Edition

Small local models love to add extra prose. Agents cannot. This lab teaches the pattern every later framework uses: **schema → prompt → parse → retry**.


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [3]:
import os

for root, dirs, files in os.walk("/content/shared"):
    for file in files:
        print(os.path.join(root, file))

/content/shared/shared/__init__.py
/content/shared/shared/openai_compat_server.py
/content/shared/shared/course_runtime.py


In [11]:
import shutil
import os

# Move all files from /content/shared/shared -> /content/shared
source = "/content/shared/shared"
destination = "/content/shared"

for item in os.listdir(source):
    shutil.move(os.path.join(source, item), destination)

# Remove the empty inner folder
os.rmdir(source)

print("✅ Folder structure fixed!")

✅ Folder structure fixed!


In [12]:
!ls -R /content/shared

/content/shared:
course_runtime.py  __init__.py	openai_compat_server.py


In [10]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Shared folder exists:", Path("/content/shared").exists())
print("course_runtime.py exists:", Path("/content/shared/course_runtime.py").exists())

# Show what's inside shared
!ls -R /content/shared

Current folder: /content
Shared folder exists: True
course_runtime.py exists: False
/content/shared:
shared

/content/shared/shared:
course_runtime.py  __init__.py	openai_compat_server.py


## 1. Setup


In [13]:
WEEK = 'Week 1'
LAB = 'Lab 2 — structured output'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
         if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 1 / Lab 2 — structured output
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [14]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pydantic
else:
    %pip install -q ollama pydantic


## 2. Why free text is not enough

Ask the model for a classification. You will often get a paragraph, markdown, or extra keys. Downstream Python code then crashes.


In [15]:
from pydantic import BaseModel, Field, ValidationError

raw = local_chat([
    {"role": "user", "content": "Extract name, age, and city from: 'Maya is 29 and lives in Pune.'"}
], max_new_tokens=80)
print(raw)


Loading Hugging Face model Qwen/Qwen2.5-1.5B-Instruct on GPU ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Name: Maya
Age: 29
City: Pune


## 3. Define the contract with Pydantic

Pydantic is the schema. The model does not 'know' Pydantic — **your loop** enforces it.


In [16]:
class Person(BaseModel):
    name: str
    age: int = Field(ge=0, le=120)
    city: str

print(Person.model_json_schema())


{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'age': {'maximum': 120, 'minimum': 0, 'title': 'Age', 'type': 'integer'}, 'city': {'title': 'City', 'type': 'string'}}, 'required': ['name', 'age', 'city'], 'title': 'Person', 'type': 'object'}


## 4. Constrained prompt + parse + retry


In [19]:
SCHEMA_PROMPT = """Return ONLY valid JSON matching this schema, no markdown:
{{"name": string, "age": integer, "city": string}}

Text: {text}
"""

def structured_person(text: str, attempts: int = 3) -> Person:
    messages = [{"role": "user", "content": SCHEMA_PROMPT.format(text=text)}]
    last_err = None
    for i in range(attempts):
        reply = local_chat(messages, max_new_tokens=80, temperature=0.1)
        obj = extract_json_object(reply)
        try:
            if not obj:
                raise ValueError(f"no JSON in: {reply!r}")
            return Person.model_validate(obj)
        except (ValidationError, ValueError) as err:
            last_err = err
            messages.append({"role": "assistant", "content": reply})
            messages.append({
                "role": "user",
                "content": f"That was invalid ({err}). Return ONLY the JSON object.",
            })
            print(f"retry {i+1}: {err}")
    raise RuntimeError(f"failed after {attempts} attempts: {last_err}")

person = structured_person("Maya is 29 and lives in Pune.")
print(person)
print(person.model_dump())


[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


name='Maya' age=29 city='Pune'
{'name': 'Maya', 'age': 29, 'city': 'Pune'}


## 5. Exercise

1. Add a `sentiment` field (`positive` | `negative` | `neutral`) and classify: *The lab was hard but I learned a lot.*
2. Feed garbage text (`asdf`) and watch retries fail — then add a fallback `Person(name='unknown', age=0, city='unknown')`.
3. Compare `temperature=0` vs `0.9` for JSON reliability.

**Next:** `lab3_tools_from_scratch.ipynb` — the model chooses a Python function to run.
